# Other organisms — mouse and pig showcase

This notebook provides a small **non-human** showcase to demonstrate that IDTrack is not limited to human.

## Rationale

The core IDTrack idea (Ensembl history as a time axis + snapshot-bounded mapping + explicit ambiguity) applies to any Ensembl-supported organism.
This experiment intentionally stays **within-species** (no ortholog mapping) and answers:

- Can we build/load the organism graph snapshot from the shared cache?
- Do we see the same 1→0 / 1→1 / 1→n outcome semantics?

This is a marketing appendix: it shows portability without diluting the main human-focused narrative.

Outputs:
- `idtrack-manuscript/figures/fig_other_organisms_outcomes.pdf`

Caching:
- Results are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/other_organisms/`.
- If caches are missing, the notebook computes them (graph loading can be memory-intensive).

Related:
- Cross-species harmonization into human is covered separately in `idtrack/docs/_notebooks/06_tutorial_humanization_mouse_pig_to_human.ipynb`.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    apply_rcparams,
    experiments_cache_dir,
    idtrack_cache_dir,
    load_rcparams,
    manuscript_figures_dir,
    read_pickle,
    write_pickle,
)

try:
    apply_rcparams(load_rcparams())
except Exception as e:  # noqa: S110
    print('Warning: could not apply shared rcParams:', e)

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
CACHE_DIR = experiments_cache_dir(REPO_ROOT, experiment='other_organisms')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

# Demo settings
N_SAMPLE_IDS = 200
STRATEGY = 'all'   # exposes ambiguity (1→n)
TO_RELEASE = 110   # choose a reasonable mid-range release for the demo

ORGANISMS = [
    ('mouse', 'ENSMUSG'),
    ('pig', 'ENSSSCG'),
]

RESULTS_PKL = CACHE_DIR / f"other_organisms_summary_n{N_SAMPLE_IDS}_to{TO_RELEASE}_strategy{STRATEGY}.pickle"

print('RESULTS_PKL:', RESULTS_PKL)


In [ ]:
# -------------------- Compute (cache-first) --------------------

import numpy as np
import pandas as pd

if RESULTS_PKL.exists():
    df = read_pickle(RESULTS_PKL)
    print('Loaded:', RESULTS_PKL)
else:
    import idtrack

    rng = np.random.default_rng(0)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    def reservoir_sample(nodes, prefix: str, k: int) -> list[str]:
        sample: list[str] = []
        n_seen = 0
        for node in nodes:
            if not isinstance(node, str) or not node.startswith(prefix):
                continue
            n_seen += 1
            if len(sample) < k:
                sample.append(node)
                continue
            j = int(rng.integers(0, n_seen))
            if j < k:
                sample[j] = node
        return sample

    rows = []
    for organism_alias, prefix in ORGANISMS:
        organism, latest = api.resolve_organism(organism_alias)
        snapshot = latest
        api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=False)

        sample = reservoir_sample(api.track.graph.nodes, prefix, N_SAMPLE_IDS)
        if not sample:
            print('No IDs found for', organism_alias, 'with prefix', prefix)
            continue

        to_release = min(int(TO_RELEASE), int(latest))

        matchings = api.convert_identifier_multiple(
            sample,
            to_release=to_release,
            final_database=None,
            strategy=STRATEGY,
            verbose=True,
            pbar_prefix=f"{organism_alias}",
        )

        bins = api.classify_multiple_conversion(matchings)
        rows.append(
            {
                'organism': organism_alias,
                'snapshot_release': int(latest),
                'to_release': int(to_release),
                'n': int(len(sample)),
                '1_to_0': int(len(bins['matching_1_to_0'])),
                '1_to_1': int(len(bins['matching_1_to_1']) + len(bins['alternative_target_1_to_1'])),
                '1_to_n': int(len(bins['matching_1_to_n']) + len(bins['alternative_target_1_to_n'])),
            }
        )

    df = pd.DataFrame(rows)
    write_pickle(df, RESULTS_PKL)
    print('Saved:', RESULTS_PKL)

df


In [ ]:
# -------------------- Plot outcome profiles --------------------

if df is None or df.empty:
    print('No results to plot.')
else:
    frac = df.set_index('organism')[['1_to_0', '1_to_1', '1_to_n']].div(df.set_index('organism')['n'], axis=0)

    fig, ax = plt.subplots(1, 1, figsize=(5.5, 3.5))
    frac.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
    )
    ax.set_ylabel('Fraction of queries')
    ax.set_xlabel('Organism')
    ax.set_ylim(0, 1)
    ax.set_title('IDTrack outcome profile (non-human showcase)')
    ax.legend(['1→0', '1→1', '1→n'], loc='upper right')

    fig.tight_layout()
    out_fig = MANUSCRIPT_FIGURES / 'fig_other_organisms_outcomes.pdf'
    fig.savefig(out_fig, bbox_inches='tight')
    print('Saved:', out_fig)


In [ ]:
# Notes

# - For cross-species harmonization into human, see `idtrack/docs/_notebooks/06_tutorial_humanization_mouse_pig_to_human.ipynb`.
# - This notebook intentionally stays within-species (no ortholog mapping).
